## Step 1

In [1]:
# Install the Google ADK
!pip install google-adk -q

## Step 2

In [2]:
# 2. Imports
import os
import vertexai
from google.adk.agents import Agent
from google.adk.tools import AgentTool, google_search
from vertexai.preview.reasoning_engines import AdkApp
from vertexai import agent_engines

# 3. Project Configuration
PROJECT_ID = "qwiklabs-gcp-04-cc20aba4b1cc" # Your project ID
LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-adk-staging"

# Initialize Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET
)

# Create the staging bucket (Required for deployment)
!gsutil mb -l {LOCATION} {STAGING_BUCKET} || true

Creating gs://qwiklabs-gcp-04-cc20aba4b1cc-adk-staging/...


## Step 3

In [3]:
# Define Specialized Agents
greeter = Agent(
    name="Greeter",
    instruction="Greet the user and explain that the research team is starting.",
    model="gemini-2.0-flash"
)

searcher = Agent(
    name="Searcher",
    instruction="Use Google Search to find current facts about the user's query.",
    tools=[google_search],
    model="gemini-2.0-flash"
)

critique = Agent(
    name="Critique",
    instruction="Evaluate search results for gaps or errors. Suggest improvements.",
    model="gemini-2.0-flash"
)

refiner = Agent(
    name="Refiner",
    instruction="Write the final, polished answer based on search data and critique.",
    model="gemini-2.0-flash"
)

# The Orchestrator Agent (Master)
workflow_orchestrator = Agent(
    name="WorkflowManager",
    instruction="""Follow this exact sequence:
    1. Call Greeter.
    2. Call Searcher.
    3. Call Critique.
    4. Call Refiner for the final answer.""",
    tools=[
        AgentTool(agent=greeter),
        AgentTool(agent=searcher),
        AgentTool(agent=critique),
        AgentTool(agent=refiner)
    ],
    model="gemini-2.0-flash"
)

## Step 4 (Local Test)

In [11]:
print("--- TEST 1: LOCAL ADKAPP VALIDATION ---")
app = AdkApp(agent=workflow_orchestrator)

query = "What is the current status of the St. Louis Cardinals baseball team?"
print("Processing local request (Streaming Dictionaries)...")

responses = app.stream_query(
    user_id="test-user-local",
    message=query
)

for event in responses:
    # 1. Handle Dictionary Response
    if isinstance(event, dict):
        content = event.get('content', {})
        parts = content.get('parts', [])
        for part in parts:
            if 'text' in part:
                print(part['text'], end="", flush=True)
            if 'function_call' in part:
                print(f"\n[SYSTEM]: Calling -> {part['function_call']['name']}\n")

    # 2. Handle Object Response (Backup)
    elif hasattr(event, 'content'):
        for part in event.content.parts:
            if hasattr(part, 'text') and part.text:
                print(part.text, end="", flush=True)
            if hasattr(part, 'function_call') and part.function_call:
                print(f"\n[SYSTEM]: Calling -> {part.function_call.name}\n")

--- TEST 1: LOCAL ADKAPP VALIDATION ---
Processing local request (Streaming Dictionaries)...


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()



[SYSTEM]: Calling -> Greeter


[SYSTEM]: Calling -> Searcher


[SYSTEM]: Calling -> Critique


[SYSTEM]: Calling -> Refiner

The St. Louis Cardinals are currently in a re-tooling phase, striving to balance competitiveness in the present with a focus on developing their future talent. This involves integrating younger players from their improving farm system, which boasts promising prospects like JJ Wetherholt and Masyn Winn. As they navigate Spring Training, the team is also managing injuries to key players, including Lars Nootbaar and Riley O'Brien. The organization's emphasis on prospects, coupled with Chaim Bloom's support for manager Oli Marmol, may help to alleviate some fan concerns about the direction of the team.


## Step 5 (Deploy)

In [12]:
print("--- TEST 2: DEPLOYING TO AGENT ENGINE ---")

# Deploy the app to the cloud
remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]", "requests"],
    display_name="deployment_agent_engine_1"
)

print(f"\nDeployment Successful!")
print(f"Resource Name: {remote_agent.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-cc20aba4b1cc-adk-staging


--- TEST 2: DEPLOYING TO AGENT ENGINE ---


INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-cc20aba4b1cc-adk-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-cc20aba4b1cc-adk-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-cc20aba4b1cc-adk-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/608565707111/locations/us-central1/reasoningEngines/6875280293343789056/operations/8174728635932999680
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-04-cc20aba4b1cc
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/608565707111/locations/us-central1/reasoningEngines/6875280293343789056
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
INFO:ver


Deployment Successful!
Resource Name: projects/608565707111/locations/us-central1/reasoningEngines/6875280293343789056


## Step 6 (Remote Test)

In [13]:
print("--- TEST 3: REMOTE INSTANCE VERIFICATION ---")

# Define a fresh query for the cloud agent
remote_query = "What have the goals of the St. Louis Cardinals been this offseason, what changes have they made, and who is on the team?"

print("Sending request to Google Cloud Agent Engine...")

try:
    # We call stream_query on the 'remote_agent' object created in the last step
    for event in remote_agent.stream_query(
        user_id="final-test-user",
        message=remote_query
    ):
        # The Remote Agent returns events as Dictionaries
        if isinstance(event, dict):
            content = event.get('content', {})
            parts = content.get('parts', [])
            for part in parts:
                if 'text' in part:
                    print(part['text'], end="", flush=True)
                if 'function_call' in part:
                    # This shows the orchestration happening LIVE in the cloud
                    print(f"\n[CLOUD DELEGATION]: {part['function_call']['name']} is active...\n")
except Exception as e:
    print(f"\nError communicating with cloud agent: {e}")

print("\n\n--- CHALLENGE FIVE COMPLETE ---")

--- TEST 3: REMOTE INSTANCE VERIFICATION ---
Sending request to Google Cloud Agent Engine...

[CLOUD DELEGATION]: Greeter is active...


[CLOUD DELEGATION]: Searcher is active...


[CLOUD DELEGATION]: Critique is active...


[CLOUD DELEGATION]: Refiner is active...

The St. Louis Cardinals are undergoing a rebuild, prioritizing youth and future potential. Having lost veteran players in free agency, the team is actively reshaping its roster with an emphasis on starting pitching, demonstrated by the acquisitions of Dustin May, Richard Fitts, and Hunter Dobbins. A key element of this rebuild is the development of young players already within the organization. Masyn Winn is poised for a larger role, with his speed and on-base skills potentially making him a leadoff hitter. JJ Wetherholt, a highly touted prospect, is expected to compete for a starting position and could be a Rookie of the Year contender. The Cardinals' Spring Breakout roster showcases the depth of their farm system, featuri